# Dasar Machine Learning

## Course 3 Bagian c: Support Vector Machines

Data yang dipakai adalah kualitas anggur: kadar asam, gula, pH, alkohol, nilai kualitas, dan warna. Warna anggur menjadi target klasifikasi.

### Tujuan

- Menyiapkan fitur dari dua kolom dengan korelasi tertinggi.
- Menggambar batas keputusan `LinearSVC`.
- Membandingkan pengaruh `gamma` dan `C` pada kernel Gaussian.
- Membandingkan waktu latih SVC dengan pendekatan Nystroem dan SGD.


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Bagian 1: Menyiapkan data

Target `y` bernilai 1 untuk anggur merah dan 0 untuk anggur putih. Korelasi setiap fitur terhadap target dihitung, lalu dua fitur dengan korelasi absolut terbesar dipakai untuk menggambar batas keputusan dua dimensi.


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/Wine_Quality_Data.csv"
)

wine_data = pd.read_csv(DATA_URL, sep=",")
is_red = (wine_data["color"] == "red").astype(int)
feature_names = list(wine_data.columns[:-1])
correlations = wine_data[feature_names].corrwith(is_red).sort_values()
correlations


In [ ]:
sns.set_context("talk")
sns.set_style("white")

sampled = wine_data.sample(400, random_state=42)
sns.pairplot(sampled, hue="color")
plt.show()


In [ ]:
axis = correlations.plot(kind="bar", figsize=(10, 6))
axis.set(ylim=[-1, 1], ylabel="pearson correlation")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler

selected_features = correlations.map(abs).sort_values().iloc[-2:].index
print(selected_features)

scaler = MinMaxScaler()
scaled_features = pd.DataFrame(
    scaler.fit_transform(wine_data[selected_features]),
    columns=[f"{name}_scaled" for name in selected_features],
)
scaled_features.head()


## Bagian 2: Batas keputusan linear

Model `LinearSVC` dilatih pada dua fitur hasil penstandaran. Sampel 300 baris dipakai untuk memperjelas sebaran titik, sedangkan seluruh area grid diprediksi untuk mewarnai batas keputusan.


In [ ]:
from sklearn.svm import LinearSVC

linear_model = LinearSVC().fit(scaled_features, is_red)

sample_rows = scaled_features.sample(300, random_state=45)
sample_colors = is_red.loc[sample_rows.index].map({1: "red", 0: "yellow"})

axis = plt.axes()
axis.scatter(sample_rows.iloc[:, 0], sample_rows.iloc[:, 1],
             color=sample_colors, alpha=1)

grid_axis = np.arange(0, 1.005, .005)
xx, yy = np.meshgrid(grid_axis, grid_axis)
grid_points = pd.DataFrame([xx.ravel(), yy.ravel()]).T
grid_prediction = linear_model.predict(grid_points).reshape(xx.shape)
axis.contourf(xx, yy, grid_prediction, cmap=plt.cm.autumn_r, alpha=.3)
axis.set(xlabel=selected_features[0], ylabel=selected_features[1],
         xlim=[0, 1], ylim=[0, 1],
         title="decision boundary for LinearSVC")
plt.show()


## Bagian 3: Kernel Gaussian

Langkah penggambaran dibungkus menjadi satu fungsi agar bisa dipakai ulang untuk estimator berbeda. Nilai `gamma` mengatur lebar pengaruh setiap titik, sedangkan `C` mengatur toleransi terhadap kesalahan klasifikasi.


In [ ]:
def plot_decision_boundary(estimator, features, target):
    estimator.fit(features, target)
    sample = features.sample(300, random_state=45)
    colors = target.loc[sample.index].map({1: "red", 0: "yellow"})

    grid_axis = np.arange(0, 1, .005)
    xx, yy = np.meshgrid(grid_axis, grid_axis)
    grid_points = pd.DataFrame([xx.ravel(), yy.ravel()]).T
    prediction = estimator.predict(grid_points).reshape(xx.shape)

    figure, axis = plt.subplots(figsize=(8, 8))
    axis.contourf(xx, yy, prediction, cmap=plt.cm.autumn_r, alpha=.3)
    axis.scatter(sample.iloc[:, 0], sample.iloc[:, 1], color=colors, alpha=1)
    axis.set(xlabel=selected_features[0], ylabel=selected_features[1],
             title=str(estimator))
    plt.show()


In [ ]:
from sklearn.svm import SVC

for gamma in [.5, 1, 2, 10]:
    plot_decision_boundary(SVC(kernel="rbf", gamma=gamma), scaled_features, is_red)


In [ ]:
for c_value in [.1, 1, 10]:
    plot_decision_boundary(
        SVC(kernel="rbf", gamma=2, C=c_value), scaled_features, is_red
    )


## Bagian 4: Perbandingan waktu latih

SVC dengan kernel RBF akurat tetapi mahal pada data besar. Alternatifnya, data dipetakan lebih dulu dengan `Nystroem`, lalu classifier linear `SGDClassifier` dilatih pada hasil pemetaan tersebut. Waktu latih keduanya diukur dengan `time.perf_counter`.


In [ ]:
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier

all_features = wine_data[wine_data.columns[:-1]]
all_target = wine_data["color"] == "red"

svc_model = SVC(kernel="rbf")
nystroem_model = Nystroem(kernel="rbf")
sgd_model = SGDClassifier()


def measure_fit_time(estimator, features, target):
    start = time.perf_counter()
    estimator.fit(features, target)
    return round(time.perf_counter() - start, 3)


def measure_approximation_time(features, target):
    start = time.perf_counter()
    transformed = nystroem_model.fit_transform(features)
    sgd_model.fit(transformed, target)
    return round(time.perf_counter() - start, 3)


print("SVC:", measure_fit_time(svc_model, all_features, all_target), "s")
print("Nystroem + SGD:", measure_approximation_time(all_features, all_target), "s")


In [ ]:
repeated_features = pd.concat([all_features] * 5)
repeated_target = pd.concat([all_target] * 5)
print(repeated_features.shape)
print(repeated_target.shape)

print("SVC:", measure_fit_time(svc_model, repeated_features, repeated_target), "s")
print("Nystroem + SGD:", measure_approximation_time(repeated_features, repeated_target), "s")


## Ringkasan

Kernel Gaussian memberi batas keputusan yang lentur, tetapi biaya latihnya tumbuh cepat seiring jumlah baris. Pendekatan Nystroem dengan classifier linear jauh lebih murah dan selisihnya makin lebar pada data besar.
